# Analyse RCEMIP — Simulation `large300`
## Transport vertical de la quantité de mouvement par la turbulence convective

**Stratégie mémoire :** un seul fichier 3D en RAM à la fois, calculs niveau par niveau.


## 1. Librairies et constantes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import xarray as xr
import gc
import os

plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})

Rd      = 287.05
Rv      = 461.5
EPSILON = Rd / Rv  # ≈ 0.622

DIR_3D = '3D'
DIR_2D = '2D'

def path3d(var):
    return os.path.join(DIR_3D, f'MESONH_RCE_large300_3D_{var}.nc')

def path2d(var):
    return os.path.join(DIR_2D, f'MESONH_RCE_large300_2D_{var}.nc')

# Taille des blocs temporels.
# RAM < 16 Go → BLOC = 1
# RAM   16 Go → BLOC = 2
# RAM   32 Go → BLOC = 5
BLOC = 2

print('Prêt.')


## 2. Lecture des dimensions (aucune donnée en RAM)

In [ ]:
# On ouvre un seul fichier juste pour lire les métadonnées
_ds = xr.open_dataset(path3d('ua'))
_da = _ds['ua']

dim_t = _da.dims[0]
dim_z = _da.dims[1]
dim_y = _da.dims[2]
dim_x = _da.dims[3]
alt    = _da[dim_z].values.copy()   # vecteur altitude [m]
n_t    = _da.sizes[dim_t]
n_z    = _da.sizes[dim_z]
n_y    = _da.sizes[dim_y]
n_x    = _da.sizes[dim_x]
times  = _da[dim_t].values.copy()

_ds.close()
del _ds, _da
gc.collect()

# Dernier tiers = état stationnaire
t_stat   = int(2 * n_t / 3)
idx_stat = slice(t_stat, None)
n_stat   = n_t - t_stat

time_days = times.astype('float64') / (1e9 * 3600 * 24)

print(f'Grille : {n_t} t  x  {n_z} z  x  {n_y} y  x  {n_x} x')
print(f'Altitude : {alt[0]:.0f} — {alt[-1]:.0f} m')
print(f'État stationnaire : t={t_stat} → {n_t-1}  ({n_stat} pas)')


## 3. PRW et auto-agrégation (fichier 2D natif)

Le fichier 2D ne contient pas la dimension verticale :  
il est léger et peut être chargé en entier sans risque.


In [ ]:
ds_prw  = xr.open_dataset(path2d('prw'))
prw_all = ds_prw['prw'].load()   # (time, y, x) — quelques centaines de Mo max
ds_prw.close()

prw_stat = prw_all.isel({dim_t: idx_stat})
prw_mean = prw_stat.mean(dim=dim_t)         # carte 2D moyenne
prw_flat = prw_mean.values.ravel()

# Variance spatiale à chaque pas de temps (signal d'agrégation)
prw_var = prw_all.var(dim=[dim_y, dim_x]).values

print(f'PRW chargée. Plage : {prw_flat.min():.1f} — {prw_flat.max():.1f} kg/m²')
print(f'Médiane : {np.median(prw_flat):.1f} kg/m²')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Carte 2D
ax = axes[0]
im = ax.pcolormesh(prw_mean.values, cmap='Blues', vmin=0)
fig.colorbar(im, ax=ax, label='PRW (kg/m²)')
ax.set_title('Carte PRW moyenne — état stationnaire', fontweight='bold')
ax.set_xlabel('x') ; ax.set_ylabel('y') ; ax.set_aspect('equal')

# Distribution
ax2 = axes[1]
med = np.median(prw_flat)
ax2.hist(prw_flat, bins=80, color='steelblue', edgecolor='white', lw=0.3)
ax2.axvline(med, color='red', lw=1.5, linestyle='--', label=f'Médiane = {med:.1f}')
ax2.set_xlabel('PRW (kg/m²)') ; ax2.set_ylabel('Colonnes')
ax2.set_title('Distribution PRW\n(bimodale → agrégation)', fontweight='bold')
ax2.legend() ; ax2.grid(True, alpha=0.3)

# Variance temporelle
ax3 = axes[2]
ax3.plot(time_days, prw_var, color='darkorange', lw=1.5)
ax3.axvline(time_days[t_stat], color='red', lw=1, linestyle='--',
            label='Début état stationnaire')
ax3.set_xlabel('Temps (jours)') ; ax3.set_ylabel('Variance PRW (kg²/m⁴)')
ax3.set_title('Variance spatiale PRW\n(croît si agrégation)', fontweight='bold')
ax3.legend() ; ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 4. Seuil sec / humide

In [ ]:
# Ajuster PRW_SEUIL selon le creux de la distribution observée au §3
PRW_SEUIL = float(np.median(prw_flat))
print(f'Seuil : {PRW_SEUIL:.1f} kg/m²')

mh = (prw_mean.values > PRW_SEUIL)   # bool (y, x)
ms = ~mh

print(f'Humide : {mh.mean()*100:.1f}%   Sec : {ms.mean()*100:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
cmap_sh = mcolors.ListedColormap(['#f5deb3', '#1a6faf'])
im = ax.pcolormesh(mh.astype(int), cmap=cmap_sh, vmin=0, vmax=1)
cbar = fig.colorbar(im, ax=ax, ticks=[0.25, 0.75])
cbar.ax.set_yticklabels(['Sec', 'Humide'])
ax.set_title(f'Carte sec / humide (seuil = {PRW_SEUIL:.1f} kg/m²)', fontweight='bold')
ax.set_xlabel('x') ; ax.set_ylabel('y') ; ax.set_aspect('equal')

ax2 = axes[1]
ax2.hist(prw_flat, bins=80, color='steelblue', edgecolor='white', lw=0.3)
ax2.axvline(PRW_SEUIL, color='black', lw=2, linestyle='--',
            label=f'Seuil = {PRW_SEUIL:.1f} kg/m²')
ax2.axvspan(0,            PRW_SEUIL,       alpha=0.1, color='saddlebrown')
ax2.axvspan(PRW_SEUIL, prw_flat.max()+1,  alpha=0.1, color='royalblue')
ax2.set_xlabel('PRW (kg/m²)') ; ax2.set_ylabel('Colonnes')
ax2.set_title('Distribution avec seuil', fontweight='bold')
ax2.legend() ; ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 5. Profil ρ₀

On charge `ta`, `pa`, `hus` **séquentiellement** (jamais simultanément)  
en accumulant les sommes nécessaires.


In [ ]:
# Accumulation de p et T·(1 + qv/ε)/(1 + qv) = Tv
# On évite de garder les trois fichiers ouverts en même temps.

rho0_sum = np.zeros(n_z)
n_rho    = 0

ds_ta  = xr.open_dataset(path3d('ta'))
ds_pa  = xr.open_dataset(path3d('pa'))
ds_hus = xr.open_dataset(path3d('hus'))

for t0 in range(t_stat, n_t, BLOC):
    t1 = min(t0 + BLOC, n_t)
    sl = {dim_t: slice(t0, t1)}

    T_blk  = ds_ta['ta'].isel(sl).values   # (bloc, z, y, x)
    p_blk  = ds_pa['pa'].isel(sl).values
    qv_blk = ds_hus['hus'].isel(sl).values

    Tv_blk  = T_blk * (1.0 + qv_blk / EPSILON) / (1.0 + qv_blk)
    rho_blk = p_blk / (Rd * Tv_blk)

    rho0_sum += rho_blk.mean(axis=(0, 2, 3)) * (t1 - t0)
    n_rho    += (t1 - t0)

    del T_blk, p_blk, qv_blk, Tv_blk, rho_blk
    gc.collect()

ds_ta.close() ; ds_pa.close() ; ds_hus.close()
del ds_ta, ds_pa, ds_hus
gc.collect()

rho0 = rho0_sum / n_rho   # profil 1D [kg/m³]

print(f'ρ₀ calculé sur {n_rho} pas de temps.')
print(f'  Surface : {rho0[0]:.3f} kg/m³')
print(f'  ~10 km  : {rho0[np.argmin(np.abs(alt-10000))]:.3f} kg/m³')

fig, ax = plt.subplots(figsize=(4, 6))
ax.plot(rho0, alt, color='steelblue', lw=2)
ax.set_xlabel('ρ₀ (kg/m³)') ; ax.set_ylabel('Altitude (m)')
ax.set_title('Profil ρ₀', fontweight='bold') ; ax.grid(True, alpha=0.3)
plt.tight_layout() ; plt.show()


## 6. Profil du vent horizontal moyen

In [ ]:
def calc_profil_vent(idx_slice):
    """Charge u et v séquentiellement, calcule le profil moyen."""
    ds_u = xr.open_dataset(path3d('ua'))
    u_m = ds_u['ua'].isel({dim_t: idx_slice}).mean(dim=[dim_t, dim_y, dim_x]).values
    ds_u.close()
    gc.collect()

    ds_v = xr.open_dataset(path3d('va'))
    v_m = ds_v['va'].isel({dim_t: idx_slice}).mean(dim=[dim_t, dim_y, dim_x]).values
    ds_v.close()
    gc.collect()

    return u_m, v_m

def plot_profil_vent(u_m, v_m, titre):
    fig, ax = plt.subplots(figsize=(5, 7))
    ax.plot(u_m,                       alt, color='royalblue', lw=2, label=r'$\overline{u}$')
    ax.plot(v_m,                       alt, color='seagreen',  lw=2, label=r'$\overline{v}$')
    ax.plot(np.sqrt(u_m**2 + v_m**2),  alt, 'k--', lw=1.5,         label='Module')
    ax.axvline(0, color='red', alpha=0.4, lw=1)
    ax.set_xlabel('Vitesse (m/s)') ; ax.set_ylabel('Altitude (m)')
    ax.set_title(titre, fontweight='bold') ; ax.grid(True, alpha=0.3) ; ax.legend()
    plt.tight_layout() ; plt.show()

u_init, v_init = calc_profil_vent(slice(0, 5))
plot_profil_vent(u_init, v_init, 'Profil du vent — état initial')

u_fin, v_fin = calc_profil_vent(idx_stat)
plot_profil_vent(u_fin, v_fin, 'Profil du vent — état stationnaire')


## 7. Bilan de QdM global

**Stratégie clé :** on ne charge jamais `u` et `w` ensemble.  
On calcule la covariance ⟨u′w′⟩ **niveau par niveau** :  
pour chaque niveau z, on extrait la tranche 2D (y, x) de u et de w,  
on calcule les anomalies, puis le produit. La tranche 2D est ~100× plus petite  
que le tableau 4D complet.

$$\text{Flux}(z) = \rho_0(z)\,\overline{u'w'}(z) = \rho_0(z) \cdot \frac{1}{N_t N_x N_y}\sum_{t,x,y} u'(t,z,x,y)\,w'(t,z,x,y)$$


In [ ]:
# ======================================================================
# Calcul niveau par niveau pour minimiser la RAM
# Pour chaque niveau z :
#   - on lit la tranche u[:, z, :, :] et w[:, z, :, :] sur l'état stationnaire
#   - on calcule u' * w' et on moyenne sur (t, y, x)
# ======================================================================

flux_uw = np.zeros(n_z)   # ρ₀ <u'w'>  [kg m⁻¹ s⁻²]
flux_vw = np.zeros(n_z)

ds_u = xr.open_dataset(path3d('ua'))
ds_v = xr.open_dataset(path3d('va'))
ds_w = xr.open_dataset(path3d('wa'))

for iz in range(n_z):
    # Tranche (t_stat, y, x) pour ce niveau
    u_lev = ds_u['ua'].isel({dim_t: idx_stat, dim_z: iz}).values  # (n_stat, ny, nx)
    w_lev = ds_w['wa'].isel({dim_t: idx_stat, dim_z: iz}).values

    # Anomalies spatiales (décomposition de Reynolds)
    u_p = u_lev - u_lev.mean(axis=(1, 2), keepdims=True)
    w_p = w_lev - w_lev.mean(axis=(1, 2), keepdims=True)
    flux_uw[iz] = rho0[iz] * (u_p * w_p).mean()

    del u_lev, w_lev, u_p, w_p

    # v'w' au même niveau
    v_lev = ds_v['va'].isel({dim_t: idx_stat, dim_z: iz}).values
    w_lev = ds_w['wa'].isel({dim_t: idx_stat, dim_z: iz}).values
    v_p = v_lev - v_lev.mean(axis=(1, 2), keepdims=True)
    w_p = w_lev - w_lev.mean(axis=(1, 2), keepdims=True)
    flux_vw[iz] = rho0[iz] * (v_p * w_p).mean()

    del v_lev, w_lev, v_p, w_p
    gc.collect()

    if iz % 10 == 0:
        print(f'  Niveau {iz}/{n_z-1}  ({alt[iz]:.0f} m)')

ds_u.close() ; ds_v.close() ; ds_w.close()
del ds_u, ds_v, ds_w
gc.collect()

print('Flux globaux calculés.')


In [ ]:
def tendance(flux_profil):
    """Force de Reynolds : -1/ρ₀ · d(ρ₀ <φ'w'>) / dz"""
    return -np.gradient(flux_profil, alt) / rho0

tend_u_glob = tendance(flux_uw)
tend_v_glob = tendance(flux_vw)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7), sharey=True)

ax1.plot(flux_uw, alt, color='royalblue', lw=2, label=r"$\rho_0\overline{u'w'}$")
ax1.plot(flux_vw, alt, color='seagreen',  lw=2, linestyle='--', label=r"$\rho_0\overline{v'w'}$")
ax1.axvline(0, color='grey', alpha=0.5)
ax1.set_xlabel('Flux ρ₀⟨φ′w′⟩  (kg m⁻¹ s⁻²)') ; ax1.set_ylabel('Altitude (m)')
ax1.set_title('A. Flux vertical de QdM', fontweight='bold')
ax1.legend() ; ax1.grid(True, alpha=0.3)

ax2.plot(tend_u_glob * 1e5, alt, color='royalblue', lw=2, label='Tendance u')
ax2.plot(tend_v_glob * 1e5, alt, color='seagreen',  lw=2, linestyle='--', label='Tendance v')
ax2.axvline(0, color='grey', alpha=0.5)
ax2.set_xlabel('Tendance (×10⁻⁵ m/s²)')
ax2.set_title('B. Tendance du vent par la turbulence\n'
              r'$-\frac{1}{\rho_0}\frac{\partial}{\partial z}(\rho_0\overline{\phi\'w\'})$',
              fontweight='bold')
ax2.legend() ; ax2.grid(True, alpha=0.3)

plt.suptitle('Bilan global de QdM — état stationnaire', fontsize=13, fontweight='bold')
plt.tight_layout() ; plt.show()


## 8. Bilan conditionnel — décomposition par régime turbulent

Même stratégie niveau par niveau, mais on a besoin de `w`, `u`, `clw`, `cli`  
pour les masques. On les lit **séquentiellement** pour chaque niveau.

| Régime | Condition |
|--------|-----------|
| Convection nuageuse | w′ > +SEUIL_W et qc > SEUIL_QC |
| Thermiques secs | w′ > +SEUIL_W et qc ≤ SEUIL_QC |
| Subsidence | w′ < −SEUIL_W |
| Petite turbulence | résidu |


In [ ]:
SEUIL_W  = 0.05   # m/s
SEUIL_QC = 1e-5   # kg/kg

flux_conv = np.zeros(n_z)
flux_sec  = np.zeros(n_z)
flux_sub  = np.zeros(n_z)
flux_tot_cond = np.zeros(n_z)

ds_u   = xr.open_dataset(path3d('ua'))
ds_w   = xr.open_dataset(path3d('wa'))
ds_clw = xr.open_dataset(path3d('clw'))
ds_cli = xr.open_dataset(path3d('cli'))

for iz in range(n_z):
    sl = {dim_t: idx_stat, dim_z: iz}

    u_lev   = ds_u['ua'].isel(sl).values    # (n_stat, ny, nx)
    w_lev   = ds_w['wa'].isel(sl).values
    clw_lev = ds_clw['clw'].isel(sl).values
    cli_lev = ds_cli['cli'].isel(sl).values

    u_p = u_lev - u_lev.mean(axis=(1, 2), keepdims=True)
    w_p = w_lev - w_lev.mean(axis=(1, 2), keepdims=True)
    qc  = clw_lev + cli_lev

    flux_loc = rho0[iz] * u_p * w_p   # (n_stat, ny, nx)

    m_conv = (w_p >  SEUIL_W) & (qc >  SEUIL_QC)
    m_sec  = (w_p >  SEUIL_W) & (qc <= SEUIL_QC)
    m_sub  = (w_p < -SEUIL_W)

    flux_conv[iz]     = np.where(m_conv, flux_loc, 0.0).mean()
    flux_sec[iz]      = np.where(m_sec,  flux_loc, 0.0).mean()
    flux_sub[iz]      = np.where(m_sub,  flux_loc, 0.0).mean()
    flux_tot_cond[iz] = flux_loc.mean()

    del u_lev, w_lev, clw_lev, cli_lev, u_p, w_p, qc, flux_loc
    del m_conv, m_sec, m_sub
    gc.collect()

    if iz % 10 == 0:
        print(f'  Niveau {iz}/{n_z-1}  ({alt[iz]:.0f} m)')

ds_u.close() ; ds_w.close() ; ds_clw.close() ; ds_cli.close()
del ds_u, ds_w, ds_clw, ds_cli
gc.collect()

flux_rest_cond = flux_tot_cond - (flux_conv + flux_sec + flux_sub)

print('Flux conditionnels calculés.')


In [ ]:
tend_conv = tendance(flux_conv)
tend_sec  = tendance(flux_sec)
tend_sub  = tendance(flux_sub)
tend_rest = tendance(flux_rest_cond)
tend_tot  = tendance(flux_tot_cond)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 8), sharey=True)

kw = dict(tot  = dict(color='black',      lw=2.5),
          conv = dict(color='crimson',    lw=1.8, linestyle='--'),
          sec  = dict(color='darkorange', lw=1.8, linestyle='--'),
          sub  = dict(color='royalblue',  lw=1.8, linestyle=':'),
          rest = dict(color='grey',       lw=1.5, alpha=0.7))

ax1.plot(flux_tot_cond,  alt, label='Total',                    **kw['tot'])
ax1.plot(flux_conv,      alt, label='Convection nuageuse',       **kw['conv'])
ax1.plot(flux_sec,       alt, label='Thermiques secs (CBL)',     **kw['sec'])
ax1.plot(flux_sub,       alt, label='Subsidence',                **kw['sub'])
ax1.plot(flux_rest_cond, alt, label='Petite turbulence (reste)', **kw['rest'])
ax1.axvline(0, color='grey', alpha=0.4)
ax1.set_title('A. Flux ρ₀⟨u′w′⟩ par régime', fontweight='bold')
ax1.set_xlabel('Flux (kg m⁻¹ s⁻²)') ; ax1.set_ylabel('Altitude (m)')
ax1.legend(fontsize=9) ; ax1.grid(True, alpha=0.3)

sc = 1e5
ax2.plot(tend_tot  * sc, alt, label='Total',                    **kw['tot'])
ax2.plot(tend_conv * sc, alt, label='Convection nuageuse',       **kw['conv'])
ax2.plot(tend_sec  * sc, alt, label='Thermiques secs (CBL)',     **kw['sec'])
ax2.plot(tend_sub  * sc, alt, label='Subsidence',                **kw['sub'])
ax2.plot(tend_rest * sc, alt, label='Petite turbulence (reste)', **kw['rest'])
ax2.axvline(0, color='grey', alpha=0.4)
ax2.set_title('B. Tendance vent zonal par régime\n'
              r'$-\frac{1}{\rho_0}\frac{\partial}{\partial z}'
              r'(\rho_0\overline{u\'w\'})$  (×10⁻⁵ m/s²)', fontweight='bold')
ax2.set_xlabel('Tendance (×10⁻⁵ m/s²)')
ax2.legend(fontsize=9) ; ax2.grid(True, alpha=0.3)

plt.suptitle('Bilan conditionnel de QdM — état stationnaire', fontsize=13, fontweight='bold')
plt.tight_layout() ; plt.show()


## 9. Bilan de QdM par région (sec / humide)

In [ ]:
flux_h = np.zeros(n_z)
flux_s = np.zeros(n_z)

ds_u = xr.open_dataset(path3d('ua'))
ds_w = xr.open_dataset(path3d('wa'))

for iz in range(n_z):
    sl = {dim_t: idx_stat, dim_z: iz}

    u_lev = ds_u['ua'].isel(sl).values   # (n_stat, ny, nx)
    w_lev = ds_w['wa'].isel(sl).values

    u_p = u_lev - u_lev.mean(axis=(1, 2), keepdims=True)
    w_p = w_lev - w_lev.mean(axis=(1, 2), keepdims=True)
    flux_loc = rho0[iz] * u_p * w_p   # (n_stat, ny, nx)

    # mh / ms sont des masques 2D (ny, nx)
    # on les broadcast sur l'axe temps
    flux_h[iz] = flux_loc[:, mh].mean()
    flux_s[iz] = flux_loc[:, ms].mean()

    del u_lev, w_lev, u_p, w_p, flux_loc
    gc.collect()

    if iz % 10 == 0:
        print(f'  Niveau {iz}/{n_z-1}')

ds_u.close() ; ds_w.close()
del ds_u, ds_w
gc.collect()

tend_h = tendance(flux_h)
tend_s = tendance(flux_s)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7), sharey=True)

ax1.plot(flux_uw, alt, 'k-',  lw=2.5, label='Global')
ax1.plot(flux_h,  alt, color='royalblue',   lw=2, linestyle='--', label='Humide')
ax1.plot(flux_s,  alt, color='saddlebrown', lw=2, linestyle=':',  label='Sec')
ax1.axvline(0, color='grey', alpha=0.4)
ax1.set_title('Flux ρ₀⟨u′w′⟩ — global vs régions', fontweight='bold')
ax1.set_xlabel('Flux (kg m⁻¹ s⁻²)') ; ax1.set_ylabel('Altitude (m)')
ax1.legend() ; ax1.grid(True, alpha=0.3)

ax2.plot(tend_u_glob * 1e5, alt, 'k-',  lw=2.5, label='Global')
ax2.plot(tend_h * 1e5,      alt, color='royalblue',   lw=2, linestyle='--', label='Humide')
ax2.plot(tend_s * 1e5,      alt, color='saddlebrown', lw=2, linestyle=':',  label='Sec')
ax2.axvline(0, color='grey', alpha=0.4)
ax2.set_title('Tendance — global vs régions (×10⁻⁵ m/s²)', fontweight='bold')
ax2.set_xlabel('Tendance (×10⁻⁵ m/s²)')
ax2.legend() ; ax2.grid(True, alpha=0.3)

plt.suptitle('Bilan de QdM : régions sèches vs humides', fontsize=13, fontweight='bold')
plt.tight_layout() ; plt.show()


## 10. Évolution temporelle du flux de Reynolds intégré

On intègre ρ₀⟨u′w′⟩ verticalement (0–15 km) à chaque pas de temps.  
Ici on traite **un pas de temps à la fois** — un seul niveau de mémoire.


In [ ]:
idx_tropo = np.searchsorted(alt, 15000)
alt_tropo = alt[:idx_tropo]
rho0_tropo = rho0[:idx_tropo]

flux_int_time = np.zeros(n_t)

ds_u = xr.open_dataset(path3d('ua'))
ds_w = xr.open_dataset(path3d('wa'))

for it in range(n_t):
    sl = {dim_t: it}

    u_t = ds_u['ua'].isel(sl).values   # (z, ny, nx)
    w_t = ds_w['wa'].isel(sl).values

    u_p = u_t - u_t.mean(axis=(1, 2), keepdims=True)
    w_p = w_t - w_t.mean(axis=(1, 2), keepdims=True)

    # Flux moyen horizontal à chaque niveau, intégré verticalement
    flux_prof = (rho0[:, None, None] * u_p * w_p).mean(axis=(1, 2))  # (z,)
    flux_int_time[it] = np.trapz(flux_prof[:idx_tropo], alt_tropo)

    del u_t, w_t, u_p, w_p, flux_prof
    if it % 20 == 0:
        gc.collect()
        print(f'  t={it}/{n_t-1}')

ds_u.close() ; ds_w.close()
del ds_u, ds_w
gc.collect()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(time_days, flux_int_time, color='black', lw=1.2)
ax.axvline(time_days[t_stat], color='red', lw=1.2, linestyle='--',
           label=f'Début état stationnaire (t={time_days[t_stat]:.1f} j)')
ax.set_xlabel('Temps (jours)') ; ax.set_ylabel('∫ ρ₀⟨u′w′⟩ dz  (kg/s²)')
ax.set_title('Évolution temporelle du flux de Reynolds intégré (0–15 km)', fontweight='bold')
ax.legend() ; ax.grid(True, alpha=0.3)
plt.tight_layout() ; plt.show()


## 11. Notes méthodologiques

### Stratégie mémoire
Le principe central est de **ne jamais avoir deux fichiers 3D complets en RAM simultanément**.  
Pour les calculs croisés (u′w′, masques conditionnels), on travaille **niveau par niveau** :  
à chaque itération, seule la tranche 2D `(n_stat, ny, nx)` du niveau courant est en RAM,  
soit ~`n_stat × ny × nx × 4 octets` par variable. Pour une grille 300×300×100 t,  
cela représente ~36 Mo par variable au lieu de 7.6 Go.

Le paramètre `BLOC` (cellule §1) ne contrôle plus les boucles principales —  
elles sont toutes niveau par niveau — mais reste disponible si on veut regrouper  
des lectures pour accélérer l'I/O sur certains systèmes.

### Densité ρ₀
$$\rho_0(z) = \frac{\overline{p}}{R_d\,\overline{T_v}}, \quad T_v = T\,\frac{1+q_v/\varepsilon}{1+q_v}, \quad \varepsilon = R_d/R_v \approx 0.622$$

### Tenseur de Reynolds pondéré
On utilise ρ₀⟨u′w′⟩ [kg m⁻¹ s⁻²] pour que la divergence verticale donne  
directement une tendance en m/s² :  
$$\left(\frac{\partial u}{\partial t}\right)_\text{Reynolds} = -\frac{1}{\rho_0}\frac{\partial(\rho_0\overline{u'w'})}{\partial z}$$
